In [173]:
# Import necessary libraries

In [174]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [175]:
# Fetch the dataet :
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')

In [176]:
df.shape

(569, 33)

In [177]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [178]:
# Safely drop columns if they exist
df.drop(columns=['id', 'Unnamed: 32'], inplace=True, errors='ignore')

# Split the dataset into training and testing set
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

# Scale the values
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# LabelEncode the labels
encoder = LabelEncoder()
encoder.fit(y_train)
y_train = encoder.transform(y_train)
y_test = encoder.transform(y_test)

# Convert to tensors and explicitly set dtype to float32 to avoid Double vs Float mismatch
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [179]:
# Create the Dataset class to preprocess data and call with a DataLoader later.
class MyCustomDataset(Dataset):
  def __init__(self,X,y):
    self.X = X
    self.y = y
  def __len__(self):
    return self.X.shape[0]
  def __getitem__(self,index):
    return self.X[index],self.y[index]

In [180]:
# Instantiate the custom dataset objects using the fixed tensors
training_set = MyCustomDataset(X_train_tensor, y_train_tensor)
testing_set = MyCustomDataset(X_test_tensor, y_test_tensor)

In [181]:
# Create DataLoaders with batch size to import data from dataset using DataLoader easily.
train_loader = DataLoader(training_set,batch_size=32,shuffle=True)
test_loader = DataLoader(testing_set,batch_size=32,shuffle=True)

In [182]:
# Create our Neural Network
class BreastCancerNN(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,5),
        nn.ReLU(),
        nn.Linear(5,1),
        nn.Sigmoid()
    )
  def forward(self,X):
    return self.network(X)

In [183]:
model = BreastCancerNN(X_train.shape[1])

In [184]:
# Define parameters
lr = 0.1
EPOCHS = 25

In [185]:
# Set optimizer and loss functions
optimizer_ = optim.SGD(model.parameters(),lr=lr)
loss_ = nn.BCELoss()

In [186]:
# Loop over epoch :
for epoch in range(EPOCHS):
  for batch_X,batch_y in train_loader: # So basically we are iterating over train_loader fetching the features and labels.
    y_pred = model(batch_X) # Pass the batch_features to model to forward propogate
    loss = loss_(y_pred,batch_y.view(-1,1)) # Reshape the batch_labels and calculate the loss value by passing to loss function
    optimizer_.zero_grad() # Clean any gradients of previous batch beforehand
    loss.backward() # BackPropogate the loss to calculate gradients .
    optimizer_.step() # Update parameters
  print(f'EPOCH : {epoch} | LOSS : {loss}')


EPOCH : 0 | LOSS : 0.3369371294975281
EPOCH : 1 | LOSS : 0.3719272017478943
EPOCH : 2 | LOSS : 0.09811724722385406
EPOCH : 3 | LOSS : 0.28263524174690247
EPOCH : 4 | LOSS : 0.07870534807443619
EPOCH : 5 | LOSS : 0.02228088118135929
EPOCH : 6 | LOSS : 0.029563408344984055
EPOCH : 7 | LOSS : 0.05493130907416344
EPOCH : 8 | LOSS : 0.02995642088353634
EPOCH : 9 | LOSS : 0.12365598231554031
EPOCH : 10 | LOSS : 0.07021959871053696
EPOCH : 11 | LOSS : 0.021293552592396736
EPOCH : 12 | LOSS : 0.03853529319167137
EPOCH : 13 | LOSS : 0.008974478580057621
EPOCH : 14 | LOSS : 0.012150207534432411
EPOCH : 15 | LOSS : 0.06038055941462517
EPOCH : 16 | LOSS : 0.03555811196565628
EPOCH : 17 | LOSS : 0.010067102499306202
EPOCH : 18 | LOSS : 0.011081245727837086
EPOCH : 19 | LOSS : 0.008734974078834057
EPOCH : 20 | LOSS : 0.022523269057273865
EPOCH : 21 | LOSS : 0.03163495287299156
EPOCH : 22 | LOSS : 0.10240975767374039
EPOCH : 23 | LOSS : 0.08745018392801285
EPOCH : 24 | LOSS : 0.02444211207330227


In [187]:
# Setup before evaluation for the model
model.eval() # Set model to evaluation mode
accuracy = []

In [188]:
# Make predictions
with torch.no_grad():
  for batch_X,batch_y in test_loader:
    y_pred = model(batch_X)
    y_pred = (y_pred > 0.9).float()

    batch_accuracy = (y_pred.view(-1,1) == batch_y).float().mean().item()
    accuracy.append(batch_accuracy)

  print(f'Accuracy : {(sum(accuracy)/len(accuracy)):.4f}')
  print(f'')



Accuracy : 0.5851



In [189]:
# Apply optimization algorithm